In [ ]:
import os
import sys

# Configurar directorio de trabajo
try:
    import google.colab
    IN_COLAB = True
    project_dir = '/content/TFMDS'
except ImportError:
    IN_COLAB = False
    project_dir = r'C:\Users\jmora\Documents\TFMDS'

os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

print("Directorio de trabajo:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from lib.metricas import comparar_metricas, resumen_metricas
from lib.graficos_dl import dashboard_metricas_dl, grafico_comparacion_algoritmos

print("✅ Librerías cargadas")

## Cargar todas las métricas

In [ ]:
# ============================================================================
# CARGAR TODAS LAS MÉTRICAS DE RESULTADOS
# ============================================================================

print("\n" + "="*100)
print("📂 CARGANDO MÉTRICAS DE TODOS LOS ALGORITMOS")
print("="*100)

datos_dir = Path('datos')
archivos_metricas = list(datos_dir.glob('resultados_metricas_*.csv'))

print(f"\n✅ Archivos encontrados: {len(archivos_metricas)}")

# Diccionario para almacenar todas las métricas
todas_metricas = []

for archivo in archivos_metricas:
    algoritmo = archivo.stem.replace('resultados_metricas_', '').upper()
    
    try:
        df_temp = pd.read_csv(archivo)
        print(f"   ✓ {algoritmo}: {len(df_temp)} modelos cargados")
        todas_metricas.extend(df_temp.to_dict('records'))
    except Exception as e:
        print(f"   ✗ Error cargando {archivo.name}: {str(e)}")

print(f"\n✅ Total de modelos cargados: {len(todas_metricas)}")

## Resumen consolidado de todas las métricas

In [ ]:
# ============================================================================
# RESUMEN CONSOLIDADO
# ============================================================================

print("\n" + "="*100)
print("📊 RESUMEN CONSOLIDADO DE TODOS LOS MODELOS")
print("="*100)

if todas_metricas:
    resumen_metricas(todas_metricas)
    
    # Crear DataFrame comparativo
    df_comparacion = comparar_metricas(todas_metricas, ordenar_por='RMSE')
    
    print(f"\n📈 TOP 10 MODELOS POR RMSE:")
    print(df_comparacion.head(10)[['Algoritmo', 'MAE', 'RMSE', 'R2', 'MAPE (%)']].to_string(index=False))
else:
    print("\n⚠️  No se encontraron métricas")

## Análisis por tipo de algoritmo

In [ ]:
# ============================================================================
# ANÁLISIS POR TIPO DE ALGORITMO
# ============================================================================

print("\n" + "="*100)
print("🔍 ANÁLISIS POR TIPO DE ALGORITMO")
print("="*100)

# Clasificar modelos
df_comparacion['Tipo'] = 'Otro'

# Tree-based
tree_models = ['RF', 'XGBoost', 'LightGBM', 'CatBoost']
mask_tree = df_comparacion['Algoritmo'].str.contains('|'.join(tree_models), case=False)
df_comparacion.loc[mask_tree, 'Tipo'] = 'Tree-Based'

# Deep Learning
dl_models = ['LSTM', 'GRU', 'NBEATS', 'DeepAR', 'Transformer', 'CNN']
mask_dl = df_comparacion['Algoritmo'].str.contains('|'.join(dl_models), case=False)
df_comparacion.loc[mask_dl, 'Tipo'] = 'Deep Learning'

# Estadísticas por tipo
print("\n📊 Estadísticas por tipo de algoritmo:")
print(df_comparacion.groupby('Tipo')[['MAE', 'RMSE', 'R2']].agg(['mean', 'min', 'max']).round(2))

# Mejores modelos por tipo
print("\n🏆 MEJORES MODELOS POR TIPO:")
for tipo in df_comparacion['Tipo'].unique():
    df_tipo = df_comparacion[df_comparacion['Tipo'] == tipo].head(3)
    print(f"\n{tipo}:")
    print(df_tipo[['Algoritmo', 'MAE', 'RMSE', 'R2']].to_string(index=False))

## Visualizaciones comparativas

### Comparación Top 15 modelos

In [ ]:
# Top 15 modelos
df_top15 = df_comparacion.head(15)

grafico_comparacion_algoritmos(
    df_resultados=df_top15,
    metricas=['MAE', 'RMSE', 'R2'],
    figsize=(16, 6)
)

### Comparación por tipo de algoritmo

In [ ]:
# Boxplot por tipo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribución de Métricas por Tipo de Algoritmo', fontsize=14, fontweight='bold')

metricas_plot = ['MAE', 'RMSE', 'R2']
for idx, metrica in enumerate(metricas_plot):
    ax = axes[idx]
    
    # Filtrar outliers extremos
    df_plot = df_comparacion[df_comparacion[metrica].notna()]
    
    sns.boxplot(data=df_plot, x='Tipo', y=metrica, ax=ax, palette='Set2')
    ax.set_title(metrica, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Heatmap de métricas (Top 20)

In [ ]:
# Heatmap de correlación entre métricas
df_top20 = df_comparacion.head(20)

# Seleccionar métricas numéricas
metricas_cols = ['MAE', 'RMSE', 'R2', 'MAPE (%)', 'SMAPE (%)']
df_heatmap = df_top20[['Algoritmo'] + metricas_cols].set_index('Algoritmo')

# Normalizar para visualización
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_norm = pd.DataFrame(
    scaler.fit_transform(df_heatmap),
    columns=df_heatmap.columns,
    index=df_heatmap.index
)

plt.figure(figsize=(10, 12))
sns.heatmap(df_norm, annot=False, cmap='RdYlGn_r', cbar_kws={'label': 'Valor normalizado'})
plt.title('Heatmap de Métricas (Top 20 Modelos)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Métricas', fontsize=11)
plt.ylabel('Modelos', fontsize=11)
plt.tight_layout()
plt.show()

## Dashboard final de métricas

In [ ]:
# Dashboard con los mejores modelos de cada tipo
mejores_por_tipo = df_comparacion.groupby('Tipo').first().reset_index()

# Convertir a diccionario
metricas_dict = {row['Algoritmo']: row.to_dict() for _, row in mejores_por_tipo.iterrows()}

dashboard_metricas_dl(
    metricas_dict=metricas_dict,
    titulo='Dashboard - Mejores Modelos por Categoría',
    figsize=(16, 10)
)

## Identificar el mejor modelo global

In [ ]:
# ============================================================================
# MEJOR MODELO GLOBAL
# ============================================================================

print("\n" + "="*100)
print("🏆 MEJOR MODELO GLOBAL")
print("="*100)

mejor_modelo = df_comparacion.iloc[0]

print(f"\n🥇 Algoritmo: {mejor_modelo['Algoritmo']}")
print(f"   Tipo: {mejor_modelo['Tipo']}")
print(f"\n📊 Métricas:")
print(f"   MAE:         {mejor_modelo['MAE']:.4f}")
print(f"   RMSE:        {mejor_modelo['RMSE']:.4f}")
print(f"   R²:          {mejor_modelo['R2']:.4f}")
print(f"   MAPE:        {mejor_modelo['MAPE (%)']:.2f}%")
print(f"   SMAPE:       {mejor_modelo['SMAPE (%)']:.2f}%")

# Comparar con baseline (naive forecast)
if len(df_comparacion) > 1:
    segundo_mejor = df_comparacion.iloc[1]
    mejora_rmse = ((segundo_mejor['RMSE'] - mejor_modelo['RMSE']) / segundo_mejor['RMSE']) * 100
    
    print(f"\n📈 Mejora respecto al segundo mejor modelo:")
    print(f"   Segundo: {segundo_mejor['Algoritmo']} (RMSE: {segundo_mejor['RMSE']:.4f})")
    print(f"   Mejora RMSE: {mejora_rmse:.2f}%")

print("\n" + "="*100)

## Exportar resultados consolidados

In [ ]:
# ============================================================================
# EXPORTAR RESULTADOS CONSOLIDADOS
# ============================================================================

print("\n" + "="*100)
print("💾 EXPORTANDO RESULTADOS CONSOLIDADOS")
print("="*100)

# Guardar comparación completa
df_comparacion.to_csv('datos/comparacion_todos_modelos.csv', index=False)
print("\n✅ Tabla completa guardada en: datos/comparacion_todos_modelos.csv")

# Guardar top 10
df_comparacion.head(10).to_csv('datos/top10_modelos.csv', index=False)
print("✅ Top 10 guardado en: datos/top10_modelos.csv")

# Guardar estadísticas por tipo
stats_tipo = df_comparacion.groupby('Tipo')[['MAE', 'RMSE', 'R2', 'MAPE (%)', 'SMAPE (%)']].describe()
stats_tipo.to_csv('datos/estadisticas_por_tipo.csv')
print("✅ Estadísticas por tipo guardadas en: datos/estadisticas_por_tipo.csv")

print("\n" + "="*100)
print("✅ ANÁLISIS COMPARATIVO COMPLETADO")
print("="*100)